## Make driving data for single point runs by subsetting GSWP3

In [ ]:
## Modified from Jessie Needham's script at
# https://github.com/NGEET/fates-tutorial/blob/main/tools/Make_met_drivers.ipynb

In [1]:
from scipy.stats import qmc
import numpy as np
import xarray as xr
import csv
import pandas as pd
import os
import netCDF4 as nc4
import sys
from tempfile import TemporaryFile                                                                                                                                 
import argparse                                                                                                                                                     
import shutil                                                                                                                                                       
import tempfile 
import random

In [2]:
## BIONTE / ZF2

site='ZF2'
lat = -2.63806
lon = -60.156944

if lon < 0 : 
    lon = 360 + lon
    print(lon)

299.843056


In [3]:
outpath='/pscratch/sd/j/jkowalcz/e3sm_scratch/pm-cpu/BIONTE_DATM'

### Loop through the three streams (solar, precip and tphwl) and extract the lat lon of interest - save to NGEET directory 

In [4]:
yrs = np.arange(1990,2015,1).tolist()
months = np.arange(1,13,1).tolist()
print(months)
print(yrs)

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]
[1990, 1991, 1992, 1993, 1994, 1995, 1996, 1997, 1998, 1999, 2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014]


In [7]:
for yr in yrs  :
    print(yr)
    for  mon in months : 
        
        solar_full = '/global/cfs/cdirs/e3sm/inputdata/atm/datm7/atm_forcing.datm7.GSWP3.0.5d.v2.c180716/Solar3Hrly/clmforc.GSWP3.c2011.0.5x0.5.Solr.{}-{:02d}.nc'.format(yr, mon)
        solar_full = xr.open_dataset(solar_full,  decode_times=False)

        lon_mids = solar_full.LONGXY[0,0:720]
        lat_mids = solar_full.LATIXY[0:360,0]

        abslat = np.abs(lat_mids - lat)
        abslon = np.abs(lon_mids - lon)
        c = np.maximum(abslon, abslat)

        ([xloc],[yloc]) = np.where(c == np.min(c))

        lat_new = lat_mids[yloc]
        lon_new = lon_mids[xloc]

        solar = solar_full.sel(lon=[xloc],lat=[yloc])
        solar.to_netcdf(outpath+'/{}/solar/clmforc.GSWP3.c2011.0.5x0.5.Solr.{}.{}-{:02d}.nc'.format(site,site, yr, mon))

        # precip
        precip_full = '/global/cfs/cdirs/e3sm/inputdata/atm/datm7/atm_forcing.datm7.GSWP3.0.5d.v2.c180716/Precip3Hrly/clmforc.GSWP3.c2011.0.5x0.5.Prec.{}-{:02d}.nc'.format(yr, mon)
        precip_full = xr.open_dataset(precip_full,  decode_times=False)
        
        precip = precip_full.sel(lon=[xloc],lat=[yloc])
        precip.to_netcdf(outpath+'/{}/precip/clmforc.GSWP3.c2011.0.5x0.5.Prec.{}.{}-{:02d}.nc'.format(site,site, yr, mon))
        
        # tphwl
        tphwl_full = '/global/cfs/cdirs/e3sm/inputdata/atm/datm7/atm_forcing.datm7.GSWP3.0.5d.v2.c180716/TPHWL3Hrly/clmforc.GSWP3.c2011.0.5x0.5.TPQWL.{}-{:02d}.nc'.format(yr, mon)
        tphwl_full = xr.open_dataset(tphwl_full,  decode_times=False)
        
        tphwl = tphwl_full.sel(lon=[xloc],lat=[yloc])
        tphwl.to_netcdf(outpath+'/{}/tphwl/clmforc.GSWP3.c2011.0.5x0.5.TPQWL.{}.{}-{:02d}.nc'.format(site,site, yr, mon))
        


1990
1991
1992
1993
1994
1995
1996
1997
1998
1999
2000
2001
2002
2003
2004
2005
2006
2007
2008
2009
2010
2011
2012
2013
2014


## Combine them into a single  file  per year

In [5]:
for yr in yrs  :
    print(yr)
    for  mon in months :

        solar_path = f"{outpath}/{site}/solar/clmforc.GSWP3.c2011.0.5x0.5.Solr.{site}.{yr}-{mon:02d}.nc"
        solar = xr.open_dataset(solar_path,  decode_times=False)

        precip_path = f"{outpath}/{site}/precip/clmforc.GSWP3.c2011.0.5x0.5.Prec.{site}.{yr}-{mon:02d}.nc"
        precip = xr.open_dataset(precip_path,  decode_times=False)

        tphwl_path = f"{outpath}/{site}/tphwl/clmforc.GSWP3.c2011.0.5x0.5.TPQWL.{site}.{yr}-{mon:02d}.nc"
        tphwl = xr.open_dataset(tphwl_path,  decode_times=False)


        # the solar time is different than the other two, which messes up merge. Replace solar time with precip time
        solar['time'] = precip['time']

        combined = xr.merge([solar, precip, tphwl])

        combined.to_netcdf(f"{outpath}/{site}/CLM1PT_data/{yr}-{mon:02d}.nc")

1990


ERROR 1: PROJ: proj_create_from_database: Open of /global/homes/j/jkowalcz/.conda/envs/myenv/share/proj failed


1991
1992
1993
1994
1995
1996
1997
1998
1999
2000
2001
2002
2003
2004
2005
2006
2007
2008
2009
2010
2011
2012
2013
2014
